In [1]:
import pyspark, os, sys, time, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F 
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("course-introduction-preparation").config("spark.driver.memory", "4g").config("spark.executor.memory", "4g").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/05 21:46:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Date Engineering Demo

In big data pipelines, choosing the right join algorithm is critical for performance.

- **Sort-Merge Join (Shuffle Join)**: Default strategy for joining large tables. Shuffles data across all worker nodes by join key and sorts both sides before merging (heavy network I/O and disk spilling).
- **Broadcast Join (Map-Side Join)**: Sends a copy of the small lookup table (e.g., 265 taxi zones) to memory on all worker nodes. Worker nodes join locally with **zero data shuffling** of the 52+ million trip records.

In [2]:
data_location = os.path.realpath("../big_data/nyc-tlc")

In [3]:
# 1. Load large trip dataset (all 13 Parquet files)
trips_df = spark.read.parquet(f"{data_location}/*.parquet")

# 2. Load small zone lookup dictionary (CSV)
zones_df = spark.read.csv(f"{data_location}/taxi_zone_lookup.csv", header=True, inferSchema=True)

# Create lookup DataFrames for Pickup Zone (PUZone) and Dropoff Zone (DOZone)
pu_zones = zones_df.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Zone").alias("PUZone")
)
do_zones = zones_df.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Zone").alias("DOZone")
)

print(f"Total Trip Records: {trips_df.count():,}")
print(f"Total Taxi Zones:   {zones_df.count():,}")

Total Trip Records: 52,447,491
Total Taxi Zones:   265


In [4]:
# -------------------------------------------------------------------------
# Approach 1: Sort-Merge Join (Disabling Auto-Broadcast)
# -------------------------------------------------------------------------
# Disable automatic broadcasting so Spark forces a full shuffle (Sort-Merge Join)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

start_time = time.time()

df_merged = ( trips_df 
    .join(pu_zones, on="PULocationID", how="left")
    .join(do_zones, on="DOLocationID", how="left")
            )

merge_count = df_merged.count()
merge_duration = time.time() - start_time

print(f"Sort-Merge Join Time: {merge_duration:.2f} seconds | Rows: {merge_count:,}")

[Stage 17:===================================================>    (11 + 1) / 12]

Sort-Merge Join Time: 8.14 seconds | Rows: 52,447,491


In [5]:
# -------------------------------------------------------------------------
# Approach 2: Broadcast Join (Explicit Broadcast Hint)
# -------------------------------------------------------------------------
start_time = time.time()

df_broadcast = ( trips_df 
    .join(F.broadcast(pu_zones), on="PULocationID", how="left") 
    .join(F.broadcast(do_zones), on="DOLocationID", how="left")
               )

broadcast_count = df_broadcast.count()
broadcast_duration = time.time() - start_time

print(f"Broadcast Join Time:  {broadcast_duration:.2f} seconds | Rows: {broadcast_count:,}")

Broadcast Join Time:  0.63 seconds | Rows: 52,447,491


In [6]:
print(f"Speedup: {merge_duration / broadcast_duration:.2f}x FASTER with Broadcast Join!")

Speedup: 12.97x FASTER with Broadcast Join!


In [7]:
route_counts = (
    df_broadcast
    .groupBy("PUZone", "DOZone")
    .agg(F.count("*").alias("trip_count"))
    .orderBy(F.col("trip_count").desc())
)
route_counts.show(truncate=False)

[Stage 28:===========================================>            (10 + 3) / 13]

+----------------------------+-------------------------+----------+
|PUZone                      |DOZone                   |trip_count|
+----------------------------+-------------------------+----------+
|Upper East Side South       |Upper East Side North    |326458    |
|Upper East Side North       |Upper East Side South    |280524    |
|Upper East Side South       |Upper East Side South    |231900    |
|Upper East Side North       |Upper East Side North    |215126    |
|Midtown Center              |Upper East Side South    |153427    |
|Upper East Side South       |Midtown Center           |143294    |
|Midtown Center              |Upper East Side North    |123997    |
|JFK Airport                 |JFK Airport              |115269    |
|Upper East Side South       |Midtown East             |114641    |
|Lincoln Square East         |Upper West Side South    |114487    |
|Upper West Side South       |Upper West Side North    |111628    |
|Upper West Side South       |Lincoln Square Eas

In [8]:
route_counts = (
    df_merged
    .groupBy("PUZone", "DOZone")
    .agg(F.count("*").alias("trip_count"))
    .orderBy(F.col("trip_count").desc())
)
route_counts.show(truncate=False)

[Stage 39:===========================================>            (10 + 3) / 13]

+----------------------------+-------------------------+----------+
|PUZone                      |DOZone                   |trip_count|
+----------------------------+-------------------------+----------+
|Upper East Side South       |Upper East Side North    |326458    |
|Upper East Side North       |Upper East Side South    |280524    |
|Upper East Side South       |Upper East Side South    |231900    |
|Upper East Side North       |Upper East Side North    |215126    |
|Midtown Center              |Upper East Side South    |153427    |
|Upper East Side South       |Midtown Center           |143294    |
|Midtown Center              |Upper East Side North    |123997    |
|JFK Airport                 |JFK Airport              |115269    |
|Upper East Side South       |Midtown East             |114641    |
|Lincoln Square East         |Upper West Side South    |114487    |
|Upper West Side South       |Upper West Side North    |111628    |
|Upper West Side South       |Lincoln Square Eas